In [1]:
import os
import cv2
import numpy as np

# -----------------------------
# PATHS (CHANGE ONLY THIS)
# -----------------------------
BASE_PATH = r"C:\Nexus\AI-Based Infarct Volume Estimation from Non-Contrast CT for Acute Anterior Circulation Stroke\Segmentation"

INPUT_PATH = os.path.join(BASE_PATH, "train", "images")
OUTPUT_PATH = r"C:\Nexus\brain_segmentation"

print("INPUT:", INPUT_PATH)
print("OUTPUT:", OUTPUT_PATH)


# -----------------------------
# BRAIN EXTRACTION FUNCTION
# -----------------------------
def extract_brain_safe(img):

    original = img.copy()
    h, w = original.shape

    # remove background
    _, mask = cv2.threshold(original, 20, 255, cv2.THRESH_BINARY)

    # remove skull
    _, skull = cv2.threshold(original, 200, 255, cv2.THRESH_BINARY)
    mask = cv2.subtract(mask, skull)

    # clean
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))

    # connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)

    best_label = 0
    best_score = -1

    for i in range(1, num_labels):
        x, y, bw, bh, area = stats[i]
        cx, cy = centroids[i]

        if cy < h * 0.3:
            continue

        if x == 0 or y == 0 or (x + bw) >= w or (y + bh) >= h:
            continue

        score = area + cy * 5

        if score > best_score:
            best_score = score
            best_label = i

    final_mask = np.zeros_like(mask)
    final_mask[labels == best_label] = 255

    brain = cv2.bitwise_and(original, original, mask=final_mask)

    if np.count_nonzero(brain) / brain.size < 0.05:
        return original

    return brain


# -----------------------------
# PROCESS ALL PATIENTS
# -----------------------------
patients = [p for p in os.listdir(INPUT_PATH) 
            if os.path.isdir(os.path.join(INPUT_PATH, p))]

print("Total patients:", len(patients))


for patient in patients:

    input_patient_path = os.path.join(INPUT_PATH, patient)
    output_patient_path = os.path.join(OUTPUT_PATH, patient)

    os.makedirs(output_patient_path, exist_ok=True)

    files = [f for f in os.listdir(input_patient_path)
             if f.lower().endswith((".png",".jpg",".jpeg"))]

    print(f"Processing patient {patient}: {len(files)} images")

    for file in files:

        img_path = os.path.join(input_patient_path, file)
        out_path = os.path.join(output_patient_path, file)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        brain = extract_brain_safe(img)

        cv2.imwrite(out_path, brain)

print("🔥 DONE: Brain extracted for ALL patients")

INPUT: C:\Nexus\AI-Based Infarct Volume Estimation from Non-Contrast CT for Acute Anterior Circulation Stroke\Segmentation\train\images
OUTPUT: C:\Nexus\brain_segmentation
Total patients: 25
Processing patient 049: 9 images
Processing patient 051: 19 images
Processing patient 058: 2 images
Processing patient 066: 12 images
Processing patient 068: 10 images
Processing patient 069: 2 images
Processing patient 070: 14 images
Processing patient 071: 19 images
Processing patient 072: 2 images
Processing patient 073: 6 images
Processing patient 076: 14 images
Processing patient 078: 11 images
Processing patient 079: 1 images
Processing patient 080: 11 images
Processing patient 082: 2 images
Processing patient 084: 4 images
Processing patient 085: 2 images
Processing patient 086: 7 images
Processing patient 087: 12 images
Processing patient 088: 6 images
Processing patient 090: 4 images
Processing patient 091: 7 images
Processing patient 092: 11 images
Processing patient 094: 12 images
Proces